In [3]:
from pathlib import Path
import sys
import tempfile

import duckdb


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    EMAIL_CSV,
    DEVICE_CSV,
    FILE_CSV,
    HTTP_CSV,
    LOGON_CSV,
    EMAIL_DAILY,
    DEVICE_DAILY,
    FILE_DAILY,
    HTTP_DAILY,
    LOGON_DAILY,
    USER_DAY_FILE,
)


# ============================================================
# HELPERS
# ============================================================

def db_path(path):
    return Path(path).resolve().as_posix()


def check_inputs():

    required_files = {
        "Email": EMAIL_CSV,
        "Device": DEVICE_CSV,
        "File": FILE_CSV,
        "HTTP": HTTP_CSV,
        "Logon": LOGON_CSV,
    }

    missing = [
        f"{name}: {path}"
        for name, path in required_files.items()
        if not Path(path).exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing required CERT files:\n"
            + "\n".join(missing)
        )

    print("✓ Required raw CERT files found.")


# ============================================================
# PATHS
# ============================================================

email_path = db_path(EMAIL_CSV)
device_path = db_path(DEVICE_CSV)
file_path = db_path(FILE_CSV)
http_path = db_path(HTTP_CSV)
logon_path = db_path(LOGON_CSV)

email_out = db_path(EMAIL_DAILY)
device_out = db_path(DEVICE_DAILY)
file_out = db_path(FILE_DAILY)
http_out = db_path(HTTP_DAILY)
logon_out = db_path(LOGON_DAILY)

final_out = db_path(USER_DAY_FILE)


# ============================================================
# START
# ============================================================

print("=" * 70)
print("BUILDING USER-DAY DATASET")
print("=" * 70)

print(f"\nProject root:\n{PROJECT_ROOT}")

print(f"\nFinal output:\n{USER_DAY_FILE}")

check_inputs()


# ============================================================
# DUCKDB
# ============================================================

with tempfile.TemporaryDirectory(
    prefix="fyp_duckdb_"
) as temp_dir:

    con = duckdb.connect()

    con.execute(
        "SET memory_limit='4GB'"
    )

    con.execute(
        "SET threads=4"
    )

    temp_path = (
        Path(temp_dir)
        .resolve()
        .as_posix()
    )

    con.execute(
        f"SET temp_directory='{temp_path}'"
    )


    # ========================================================
    # 1. EMAIL DAILY
    # ========================================================

    print("\n[1/6] Processing EMAIL...")

    con.execute(f"""
        COPY (

            WITH clean AS (

                SELECT
                    strptime(
                        date,
                        '%m/%d/%Y %H:%M:%S'
                    ) AS timestamp,

                    user,

                    TRY_CAST(
                        size AS BIGINT
                    ) AS email_size,

                    TRY_CAST(
                        attachments AS INTEGER
                    ) AS attachments,

                    lower(
                        concat_ws(
                            ';',
                            coalesce("to", ''),
                            coalesce(cc, ''),
                            coalesce(bcc, '')
                        )
                    ) AS recipients

                FROM read_csv(
                    '{email_path}',
                    header = true,
                    all_varchar = true
                )
            ),

            classified AS (

                SELECT
                    *,

                    regexp_replace(
                        recipients,
                        '[^;]*@dtaa[.]com',
                        '',
                        'g'
                    ) AS after_internal_removed

                FROM clean
            )

            SELECT
                user,

                CAST(
                    timestamp AS DATE
                ) AS day,

                COUNT(*) AS email_count,

                SUM(
                    CASE
                        WHEN position(
                            '@'
                            IN after_internal_removed
                        ) > 0
                        THEN 1
                        ELSE 0
                    END
                ) AS external_email_events,

                SUM(
                    coalesce(
                        email_size,
                        0
                    )
                ) AS total_email_bytes,

                SUM(
                    coalesce(
                        attachments,
                        0
                    )
                ) AS attachment_count

            FROM classified

            GROUP BY
                user,
                CAST(timestamp AS DATE)

        )
        TO '{email_out}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            OVERWRITE_OR_IGNORE TRUE
        )
    """)

    print("✓ EMAIL complete.")


    # ========================================================
    # 2. DEVICE / USB DAILY
    # ========================================================

    print("\n[2/6] Processing DEVICE...")

    con.execute(f"""
        COPY (

            WITH clean AS (

                SELECT
                    strptime(
                        date,
                        '%m/%d/%Y %H:%M:%S'
                    ) AS timestamp,

                    user,
                    activity

                FROM read_csv(
                    '{device_path}',
                    header = true,
                    all_varchar = true
                )
            )

            SELECT
                user,

                CAST(
                    timestamp AS DATE
                ) AS day,

                SUM(
                    CASE
                        WHEN activity = 'Connect'
                        THEN 1
                        ELSE 0
                    END
                ) AS usb_connect_count,

                SUM(
                    CASE
                        WHEN activity = 'Disconnect'
                        THEN 1
                        ELSE 0
                    END
                ) AS usb_disconnect_count

            FROM clean

            GROUP BY
                user,
                CAST(timestamp AS DATE)

        )
        TO '{device_out}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            OVERWRITE_OR_IGNORE TRUE
        )
    """)

    print("✓ DEVICE complete.")


    # ========================================================
    # 3. FILE COPIES TO REMOVABLE MEDIA
    # ========================================================

    print("\n[3/6] Processing FILE...")

    con.execute(f"""
        COPY (

            WITH clean AS (

                SELECT
                    strptime(
                        date,
                        '%m/%d/%Y %H:%M:%S'
                    ) AS timestamp,

                    user,
                    filename

                FROM read_csv(
                    '{file_path}',
                    header = true,
                    all_varchar = true
                )
            )

            SELECT
                user,

                CAST(
                    timestamp AS DATE
                ) AS day,

                COUNT(*) AS files_copied_to_usb,

                COUNT(
                    DISTINCT NULLIF(
                        lower(
                            regexp_extract(
                                filename,
                                '[.]([^.]+)$',
                                1
                            )
                        ),
                        ''
                    )
                ) AS unique_file_extensions

            FROM clean

            GROUP BY
                user,
                CAST(timestamp AS DATE)

        )
        TO '{file_out}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            OVERWRITE_OR_IGNORE TRUE
        )
    """)

    print("✓ FILE complete.")


    # ========================================================
    # 4. HTTP DAILY
    # ========================================================

    print("\n[4/6] Processing HTTP...")
    print(
        "This is the largest CERT file, "
        "so this step may take longer."
    )

    con.execute(f"""
        COPY (

            WITH clean AS (

                SELECT
                    strptime(
                        date,
                        '%m/%d/%Y %H:%M:%S'
                    ) AS timestamp,

                    user,

                    lower(
                        regexp_extract(
                            url,
                            'https?://([^/]+)',
                            1
                        )
                    ) AS domain

                FROM read_csv(
                    '{http_path}',
                    header = true,
                    all_varchar = true
                )
            )

            SELECT
                user,

                CAST(
                    timestamp AS DATE
                ) AS day,

                COUNT(*) AS http_request_count,

                COUNT(
                    DISTINCT domain
                ) AS unique_domains

            FROM clean

            GROUP BY
                user,
                CAST(timestamp AS DATE)

        )
        TO '{http_out}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            OVERWRITE_OR_IGNORE TRUE
        )
    """)

    print("✓ HTTP complete.")


    # ========================================================
    # 5. LOGON DAILY
    # ========================================================

    print("\n[5/6] Processing LOGON...")

    con.execute(f"""
        COPY (

            WITH clean AS (

                SELECT
                    strptime(
                        date,
                        '%m/%d/%Y %H:%M:%S'
                    ) AS timestamp,

                    user,
                    activity

                FROM read_csv(
                    '{logon_path}',
                    header = true,
                    all_varchar = true
                )
            )

            SELECT
                user,

                CAST(
                    timestamp AS DATE
                ) AS day,

                SUM(
                    CASE
                        WHEN activity = 'Logon'
                        THEN 1
                        ELSE 0
                    END
                ) AS logon_count,

                MIN(
                    CASE
                        WHEN activity = 'Logon'
                        THEN timestamp
                    END
                ) AS first_logon,

                MAX(
                    CASE
                        WHEN activity = 'Logoff'
                        THEN timestamp
                    END
                ) AS last_logoff

            FROM clean

            GROUP BY
                user,
                CAST(timestamp AS DATE)

        )
        TO '{logon_out}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            OVERWRITE_OR_IGNORE TRUE
        )
    """)

    print("✓ LOGON complete.")


    # ========================================================
    # 6. BUILD USER-DAY TABLE
    # ========================================================

    print("\n[6/6] Building USER-DAY table...")

    con.execute(f"""
        COPY (

            WITH all_activity AS (

                SELECT
                    user,
                    day
                FROM read_parquet(
                    '{email_out}'
                )

                UNION

                SELECT
                    user,
                    day
                FROM read_parquet(
                    '{device_out}'
                )

                UNION

                SELECT
                    user,
                    day
                FROM read_parquet(
                    '{file_out}'
                )

                UNION

                SELECT
                    user,
                    day
                FROM read_parquet(
                    '{http_out}'
                )

                UNION

                SELECT
                    user,
                    day
                FROM read_parquet(
                    '{logon_out}'
                )
            ),

            bounds AS (

                SELECT
                    user,
                    MIN(day) AS first_day,
                    MAX(day) AS last_day

                FROM all_activity

                GROUP BY user
            ),

            calendar AS (

                SELECT
                    b.user,

                    CAST(
                        g.day AS DATE
                    ) AS day

                FROM bounds b,

                LATERAL generate_series(
                    b.first_day,
                    b.last_day,
                    INTERVAL 1 DAY
                ) AS g(day)
            )

            SELECT
                c.user,
                c.day,

                dayofweek(
                    c.day
                ) AS day_of_week,

                CASE
                    WHEN dayofweek(
                        c.day
                    ) IN (0, 6)
                    THEN 1
                    ELSE 0
                END AS is_weekend,


                -- EMAIL
                coalesce(
                    e.email_count,
                    0
                ) AS email_count,

                coalesce(
                    e.external_email_events,
                    0
                ) AS external_email_events,

                coalesce(
                    e.total_email_bytes,
                    0
                ) AS total_email_bytes,

                coalesce(
                    e.attachment_count,
                    0
                ) AS attachment_count,


                -- DEVICE / USB
                coalesce(
                    d.usb_connect_count,
                    0
                ) AS usb_connect_count,

                coalesce(
                    d.usb_disconnect_count,
                    0
                ) AS usb_disconnect_count,


                -- FILE
                coalesce(
                    f.files_copied_to_usb,
                    0
                ) AS files_copied_to_usb,

                coalesce(
                    f.unique_file_extensions,
                    0
                ) AS unique_file_extensions,


                -- HTTP
                coalesce(
                    h.http_request_count,
                    0
                ) AS http_request_count,

                coalesce(
                    h.unique_domains,
                    0
                ) AS unique_domains,


                -- LOGON
                coalesce(
                    l.logon_count,
                    0
                ) AS logon_count,

                l.first_logon,
                l.last_logoff


            FROM calendar c


            LEFT JOIN read_parquet(
                '{email_out}'
            ) e
                USING (user, day)


            LEFT JOIN read_parquet(
                '{device_out}'
            ) d
                USING (user, day)


            LEFT JOIN read_parquet(
                '{file_out}'
            ) f
                USING (user, day)


            LEFT JOIN read_parquet(
                '{http_out}'
            ) h
                USING (user, day)


            LEFT JOIN read_parquet(
                '{logon_out}'
            ) l
                USING (user, day)


            ORDER BY
                c.user,
                c.day

        )
        TO '{final_out}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            OVERWRITE_OR_IGNORE TRUE
        )
    """)

    print("✓ USER-DAY complete.")


    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    print("\n" + "=" * 70)
    print("USER-DAY VALIDATION")
    print("=" * 70)


    # --------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------

    summary = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT user) AS users,
            MIN(day) AS first_day,
            MAX(day) AS last_day

        FROM read_parquet(
            '{final_out}'
        )
    """).df()

    print("\nDataset summary:")
    display(summary)


    # --------------------------------------------------------
    # External email validation
    # --------------------------------------------------------

    email_check = con.execute(f"""
        SELECT

            SUM(
                email_count
            ) AS total_emails,

            SUM(
                external_email_events
            ) AS external_email_events,

            ROUND(
                100.0
                * SUM(
                    external_email_events
                )
                / NULLIF(
                    SUM(email_count),
                    0
                ),
                2
            ) AS external_percentage,

            SUM(
                CASE
                    WHEN external_email_events
                         > email_count
                    THEN 1
                    ELSE 0
                END
            ) AS invalid_rows

        FROM read_parquet(
            '{final_out}'
        )
    """).df()

    print("\nExternal email validation:")
    display(email_check)


    # --------------------------------------------------------
    # Duplicate User-Day check
    # --------------------------------------------------------

    duplicates = con.execute(f"""
        SELECT COUNT(*)

        FROM (

            SELECT
                user,
                day,
                COUNT(*) AS n

            FROM read_parquet(
                '{final_out}'
            )

            GROUP BY
                user,
                day

            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    print(
        f"\nDuplicate user-day groups : "
        f"{duplicates}"
    )


    # --------------------------------------------------------
    # File extension logical check
    # --------------------------------------------------------

    invalid_extensions = con.execute(f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{final_out}'
        )

        WHERE
            unique_file_extensions
            > files_copied_to_usb
    """).fetchone()[0]

    print(
        f"Invalid file-extension rows: "
        f"{invalid_extensions}"
    )


    # --------------------------------------------------------
    # Known CERT case
    # --------------------------------------------------------

    known_case = con.execute(f"""
        SELECT
            user,
            day,
            email_count,
            external_email_events,
            usb_connect_count,
            files_copied_to_usb,
            unique_file_extensions,
            http_request_count,
            logon_count

        FROM read_parquet(
            '{final_out}'
        )

        WHERE
            user = 'AAF0535'
            AND day = DATE '2010-08-10'
    """).df()

    print(
        "\nKnown CERT case: "
        "AAF0535 / 2010-08-10"
    )

    display(known_case)


    # ========================================================
    # AUTOMATED ASSERTIONS
    # ========================================================

    rows = int(
        summary.iloc[0]["rows"]
    )

    users = int(
        summary.iloc[0]["users"]
    )

    first_day = str(
        summary.iloc[0]["first_day"]
    )

    last_day = str(
        summary.iloc[0]["last_day"]
    )


    total_emails = int(
        email_check.iloc[0][
            "total_emails"
        ]
    )

    external_emails = int(
        email_check.iloc[0][
            "external_email_events"
        ]
    )

    external_percentage = float(
        email_check.iloc[0][
            "external_percentage"
        ]
    )

    invalid_email_rows = int(
        email_check.iloc[0][
            "invalid_rows"
        ]
    )


    # Dataset size
    assert rows == 464687, (
        f"Unexpected row count: {rows}"
    )

    # Users
    assert users == 1000, (
        f"Unexpected user count: {users}"
    )

    # Date range
    assert first_day.startswith(
        "2010-01-02"
    ), (
        f"Unexpected first date: "
        f"{first_day}"
    )

    assert last_day.startswith(
        "2011-05-17"
    ), (
        f"Unexpected last date: "
        f"{last_day}"
    )

    # Email totals
    assert total_emails == 2629979, (
        f"Unexpected total email count: "
        f"{total_emails}"
    )

    assert external_emails == 1264049, (
        f"Unexpected external email count: "
        f"{external_emails}"
    )

    # External email percentage
    assert abs(
        external_percentage - 48.06
    ) < 0.01, (
        f"Unexpected external email percentage: "
        f"{external_percentage}"
    )

    # Logical consistency
    assert invalid_email_rows == 0, (
        f"Invalid external email rows: "
        f"{invalid_email_rows}"
    )

    assert duplicates == 0, (
        f"Duplicate user-day groups found: "
        f"{duplicates}"
    )

    assert invalid_extensions == 0, (
        f"Invalid file-extension rows: "
        f"{invalid_extensions}"
    )

    # Known CERT case
    assert len(known_case) == 1, (
        "AAF0535 known case was not found."
    )

    assert int(
        known_case.iloc[0][
            "files_copied_to_usb"
        ]
    ) == 11, (
        "AAF0535 expected 11 USB file copies "
        "on 2010-08-10."
    )


    # ========================================================
    # FINAL STATUS
    # ========================================================

    print("\n" + "=" * 70)
    print("✅ BUILD USER-DAY: PASS")
    print("All validation checks passed.")
    print("=" * 70)

    print(
        f"\nRows               : "
        f"{rows:,}"
    )

    print(
        f"Users              : "
        f"{users:,}"
    )

    print(
        f"Date range         : "
        f"{first_day} → {last_day}"
    )

    print(
        f"Total emails       : "
        f"{total_emails:,}"
    )

    print(
        f"External emails    : "
        f"{external_emails:,}"
    )

    print(
        f"External rate      : "
        f"{external_percentage:.2f}%"
    )

    print(
        f"Duplicate user-days: "
        f"{duplicates}"
    )

    print(
        f"Invalid extensions : "
        f"{invalid_extensions}"
    )

    print(
        f"\nSaved to:\n"
        f"{USER_DAY_FILE}"
    )

    con.close()

BUILDING USER-DAY DATASET

Project root:
C:\Users\HP\Documents\FYP\Data_engineering

Final output:
C:\Users\HP\Documents\FYP\Data_engineering\data\processed\user_day_aggregated_v1.parquet
✓ Required raw CERT files found.

[1/6] Processing EMAIL...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ EMAIL complete.

[2/6] Processing DEVICE...
✓ DEVICE complete.

[3/6] Processing FILE...
✓ FILE complete.

[4/6] Processing HTTP...
This is the largest CERT file, so this step may take longer.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ HTTP complete.

[5/6] Processing LOGON...
✓ LOGON complete.

[6/6] Building USER-DAY table...
✓ USER-DAY complete.

USER-DAY VALIDATION

Dataset summary:


,rows,users,first_day,last_day
0,464687,1000,2010-01-02,2011-05-17



External email validation:


,total_emails,external_email_events,external_percentage,invalid_rows
0,2629979.0,1264049.0,48.06,0.0



Duplicate user-day groups : 0
Invalid file-extension rows: 0

Known CERT case: AAF0535 / 2010-08-10


,user,day,email_count,external_email_events,usb_connect_count,files_copied_to_usb,unique_file_extensions,http_request_count,logon_count
0,AAF0535,2010-08-10,4,1.0,5.0,11,3,29,1.0



✅ BUILD USER-DAY: PASS
All validation checks passed.

Rows               : 464,687
Users              : 1,000
Date range         : 2010-01-02 00:00:00 → 2011-05-17 00:00:00
Total emails       : 2,629,979
External emails    : 1,264,049
External rate      : 48.06%
Duplicate user-days: 0
Invalid extensions : 0

Saved to:
C:\Users\HP\Documents\FYP\Data_engineering\data\processed\user_day_aggregated_v1.parquet


In [5]:
# ============================================================
# USER-DAY OUTPUT INSPECTION
# ============================================================

path = USER_DAY_FILE.resolve().as_posix()

con = duckdb.connect()

print("=" * 70)
print("USER-DAY DATASET INSPECTION")
print("=" * 70)


sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')
    LIMIT 10
""").df()

print("\nFirst 10 records:")
display(sample)


summary = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user) AS users,
        MIN(day) AS first_day,
        MAX(day) AS last_day
    FROM read_parquet('{path}')
""").df()

print("\nDataset summary:")
display(summary)


con.close()

print("\n✅ USER-DAY OUTPUT INSPECTION COMPLETE")

USER-DAY DATASET INSPECTION

First 10 records:


,user,day,day_of_week,is_weekend,email_count,external_email_events,total_email_bytes,attachment_count,usb_connect_count,usb_disconnect_count,files_copied_to_usb,unique_file_extensions,http_request_count,unique_domains,logon_count,first_logon,last_logoff
0,AAE0190,2010-01-04,1,0,14,2.0,441328.0,4.0,0.0,0.0,0,0,143,52,1.0,2010-01-04 08:09:00,2010-01-04 18:20:00
1,AAE0190,2010-01-05,2,0,13,7.0,355552.0,2.0,0.0,0.0,0,0,143,51,1.0,2010-01-05 08:19:00,2010-01-05 18:11:00
2,AAE0190,2010-01-06,3,0,14,6.0,532647.0,12.0,0.0,0.0,0,0,143,46,1.0,2010-01-06 08:09:00,2010-01-06 18:10:00
3,AAE0190,2010-01-07,4,0,14,5.0,474631.0,9.0,0.0,0.0,0,0,143,46,1.0,2010-01-07 08:23:00,2010-01-07 18:14:00
4,AAE0190,2010-01-08,5,0,13,3.0,400637.0,10.0,0.0,0.0,0,0,143,51,1.0,2010-01-08 08:17:00,2010-01-08 18:26:00
5,AAE0190,2010-01-09,6,1,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,NaT,NaT
6,AAE0190,2010-01-10,0,1,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,NaT,NaT
7,AAE0190,2010-01-11,1,0,13,10.0,389695.0,1.0,0.0,0.0,0,0,143,49,1.0,2010-01-11 08:09:00,2010-01-11 18:24:00
8,AAE0190,2010-01-12,2,0,14,5.0,404406.0,4.0,0.0,0.0,0,0,143,49,1.0,2010-01-12 08:12:00,2010-01-12 18:30:00
9,AAE0190,2010-01-13,3,0,14,6.0,352808.0,2.0,0.0,0.0,0,0,143,52,1.0,2010-01-13 08:15:00,2010-01-13 18:14:00



Dataset summary:


,rows,users,first_day,last_day
0,464687,1000,2010-01-02,2011-05-17



✅ USER-DAY OUTPUT INSPECTION COMPLETE


In [7]:
# ============================================================
# KNOWN CERT CASE: AAF0535
# ============================================================

path = USER_DAY_FILE.resolve().as_posix()

con = duckdb.connect()

known_user = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')

    WHERE user = 'AAF0535'

      AND day BETWEEN
          DATE '2010-07-25'
          AND DATE '2010-08-21'

    ORDER BY day
""").df()

con.close()


print("=" * 70)
print("KNOWN CERT USER: AAF0535")
print("=" * 70)

print(
    "\nObservation period : "
    "2010-07-25 → 2010-08-21"
)

print(
    f"User-Day records   : "
    f"{len(known_user)}"
)

print("\nDaily activity:")

display(known_user)

print(
    "\n✅ KNOWN USER INSPECTION COMPLETE"
)

KNOWN CERT USER: AAF0535

Observation period : 2010-07-25 → 2010-08-21
User-Day records   : 28

Daily activity:


,user,day,day_of_week,is_weekend,email_count,external_email_events,total_email_bytes,attachment_count,usb_connect_count,usb_disconnect_count,files_copied_to_usb,unique_file_extensions,http_request_count,unique_domains,logon_count,first_logon,last_logoff
0,AAF0535,2010-07-25,0,1,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,NaT,NaT
1,AAF0535,2010-07-26,1,0,3,1.0,93440.0,2.0,1.0,1.0,0,0,39,25,1.0,2010-07-26 08:56:00,2010-07-26 16:54:00
2,AAF0535,2010-07-27,2,0,3,1.0,76599.0,0.0,2.0,2.0,2,2,29,17,1.0,2010-07-27 09:01:00,2010-07-27 16:57:00
3,AAF0535,2010-07-28,3,0,3,1.0,96579.0,4.0,2.0,1.0,0,0,29,13,1.0,2010-07-28 09:06:00,2010-07-28 16:57:00
4,AAF0535,2010-07-29,4,0,3,3.0,105544.0,1.0,1.0,1.0,4,1,29,20,1.0,2010-07-29 08:58:00,2010-07-29 17:06:00
5,AAF0535,2010-07-30,5,0,3,2.0,75342.0,1.0,3.0,3.0,4,2,29,14,1.0,2010-07-30 09:01:00,2010-07-30 16:51:00
6,AAF0535,2010-07-31,6,1,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,NaT,NaT
7,AAF0535,2010-08-01,0,1,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,NaT,NaT
8,AAF0535,2010-08-02,1,0,3,3.0,91242.0,2.0,2.0,2.0,5,1,29,15,1.0,2010-08-02 08:54:00,2010-08-02 17:06:00
9,AAF0535,2010-08-03,2,0,3,2.0,105169.0,2.0,5.0,5.0,6,3,29,16,1.0,2010-08-03 08:56:00,2010-08-03 16:56:00



✅ KNOWN USER INSPECTION COMPLETE
